In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251020_214522.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/3500_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_3500_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/test_500_train_3500_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_3500_train_500_test.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id    subreddit                                              title  \
 0  1o3a9pf  miscarriage                My friend miscarried, what do I do?   
 1  1nrdmyk  miscarriage                  Irregular bleeding after 3 weeks?   
 2  1nutyvy  miscarriage               Heartbeat detected 4 days ago… gone?   
 3   cz6tt7    Parenting    Ahhh, I love the smell of logic in the morning.   
 4   lu7mr4      assault  I was the perpetrator of child-on-child sexual...   
 
                                             selftext          created_utc  \
 0  My friend lives a few hours away and had a mis...  2025-10-10T19:19:59   
 1  I had my first miscarriage  on September 1st. ...  2025-09-26T21:32:51   
 2  3 days ago (sept 27) I had my first ultrasound...  2025-10-01T00:29:25   
 3  Super quick story. \n\nMy six year old tried t...  2019-09-03 16:03:06   
 4  So, about 4 years ago, I was at my friend's ho...   2021-02-28 5:26:43   
 
                                                  url 

In [3]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
test11_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test11_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
test11_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(test11_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/110 [00:00<?, ?it/s]

In [4]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [5]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(test11_emb_A, test11_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_500_train_3500_test.json
